# BLG-407 Makine Öğrenmesi - Proje 2
## YOLOv8 ile Çatal/Kaşık Nesne Tespiti

---

| Bilgi | Değer |
|-------|-------|
| **Adı Soyadı** | İbrahim Kahraman |
| **Okul Numarası** | 2212729009 |
| **GitHub Repo** | [github.com/ibrahimkahramann/YoloV8_Object_Detection](https://github.com/ibrahimkahramann/YoloV8_Object_Detection) |

---

### Proje Özeti
Bu notebook, kendi oluşturduğum "çatal" (fork) ve "kaşık" (spoon) görüntülerinden oluşan veri seti üzerinde **YOLOv8n** modelini eğitmek için hazırlanmıştır. Eğitim sonucunda elde edilen `best.pt` dosyası, PyQt5 tabanlı bir GUI uygulamasında kullanılmaktadır.

## 1. Gerekli Kütüphanelerin İçe Aktarılması

Bu adımda, YOLOv8 modelini eğitmek için gereken temel kütüphaneler içe aktarılır:

- **ultralytics**: YOLOv8 modelini yüklemek ve eğitmek için kullanılan ana kütüphane
- **os**: Dosya yollarını yönetmek için Python standart kütüphanesi
- **torch**: PyTorch deep learning framework'ü (ultralytics arka planda kullanır)

In [1]:
from ultralytics import YOLO
import os
import torch

## 2. YOLOv8 Modelinin Yüklenmesi

Bu adımda, Ultralytics tarafından sağlanan önceden eğitilmiş **YOLOv8n** (nano) modeli yüklenir.

- **YOLOv8n**: En küçük ve en hızlı YOLOv8 varyantı (3.2M parametre)
- Model, COCO veri seti üzerinde önceden eğitilmiş ağırlıklarla başlatılır
- Transfer learning ile kendi veri setimize uyarlanacaktır

> 💡 **Not:** İlk çalıştırmada `yolov8n.pt` dosyası otomatik olarak indirilir.

In [2]:
# 1. Model Tanımı
# 'yolov8n.pt' dosyasını indirir ve başlangıç ağırlığı olarak kullanır.
model = YOLO('yolov8n.pt')

## 3. Veri Seti Konfigürasyonu (data.yaml)

`data.yaml` dosyası, YOLOv8'in veri setini tanıması için gerekli konfigürasyonu içerir:

```yaml
train: ../train/images    # Eğitim görüntülerinin yolu
val: ../valid/images      # Validasyon görüntülerinin yolu
test: ../test/images      # Test görüntülerinin yolu

nc: 2                     # Sınıf sayısı
names: ['fork', 'spoon']  # Sınıf isimleri
```

### Veri Seti Detayları
- **Toplam Görüntü Sayısı:** 370+ adet
- **Sınıflar:** Çatal (fork), Kaşık (spoon)
- **Etiketleme Aracı:** Roboflow
- **Format:** YOLOv8 uyumlu (txt dosyaları)

In [3]:
# 2. data.yaml Dosyasının Yolu
data_yaml_path = os.path.join(os.getcwd(), "data.yaml")

## 4. Model Eğitimi

Bu adımda YOLOv8 modeli, hazırladığımız veri seti üzerinde eğitilir.

### Eğitim Parametreleri
| Parametre | Değer | Açıklama |
|-----------|-------|----------|
| `epochs` | 50 | Toplam eğitim turu sayısı |
| `imgsz` | 640 | Görüntü boyutu (640x640 piksel) |
| `device` | cpu | Eğitim yapılacak cihaz (GPU için 0) |
| `batch` | 16 | Her iterasyonda işlenen görüntü sayısı |

### Beklenen Çıktılar
- **Loss Grafikleri:** Box loss, cls loss, dfl loss
- **Metrikler:** Precision, Recall, mAP50, mAP50-95
- **Model Dosyaları:** `best.pt` (en iyi), `last.pt` (son epoch)

> ⏱️ **Tahmini Süre:** CPU'da ~1-2 saat, GPU'da ~15-30 dakika

In [4]:
print(f"data.yaml yolu: {data_yaml_path}")
print("--- YOLOv8 Eğitimi Başlıyor ---")

# 3. Eğitimi Başlat
results = model.train(
    data=data_yaml_path,              # Veri seti konfigürasyon dosyası
    epochs=50,                        # Eğitim tur sayısı (Ödev için 50 ideal)
    imgsz=640,                        # Görüntü boyutu
    device='cpu',                     # İşlemci (GPU varsa 0 yazabilirsiniz, yoksa 'cpu')
    project='YoloV8_Object_Detection_Runs', # Sonuçların kaydedileceği ana klasör
    name='fork_spoon_run_new'         # Bu eğitimin klasör adı
)

print("--- Eğitim Tamamlandı ---")

data.yaml yolu: c:\Users\ibrahim\Desktop\vsc\YoloV8_Object_Detection\data.yaml
--- YOLOv8 Eğitimi Başlıyor ---
New https://pypi.org/project/ultralytics/8.3.236 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.228  Python-3.13.7 torch-2.8.0+cpu CPU (AMD Ryzen 7 6800H with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\ibrahim\Desktop\vsc\YoloV8_Object_Detection\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, 

## 5. Eğitim Sonuçları ve Model Değerlendirmesi

### Eğitim Ortamı
| Özellik | Değer |
|---------|-------|
| **İşlemci** | AMD Ryzen 7 6800H with Radeon Graphics |
| **Cihaz** | CPU |
| **Epoch Sayısı** | 50 |
| **Toplam Süre** | ~1.25 saat |
| **Model** | YOLOv8n (72 katman, 3,006,038 parametre) |
| **Framework** | Ultralytics 8.3.228, Python 3.13.7, PyTorch 2.9.1+cpu |

---

### Başarı Metrikleri (Validation Sonuçları)

Eğitim sonunda `best.pt` ağırlığı ile yapılan validasyon testi:

| Sınıf | Görüntü | Örnek | Precision | Recall | mAP50 | mAP50-95 |
|-------|---------|-------|-----------|--------|-------|----------|
| **Tümü** | 49 | 58 | 0.998 | 0.972 | 0.993 | 0.902 |
| fork | 28 | 28 | 0.996 | 0.964 | 0.991 | 0.902 |
| spoon | 30 | 30 | 1.000 | 0.979 | 0.995 | 0.901 |

---

### Metrik Açıklamaları

- **Precision (Kesinlik):** %99.8 - Model, tespit ettiği nesnelerin %99.8'inde doğru sınıfı tahmin etti
- **Recall (Duyarlılık):** %97.2 - Model, veri setindeki nesnelerin %97.2'sini başarıyla tespit etti
- **mAP50:** %99.3 - IoU eşiği 0.5'te ortalama doğruluk
- **mAP50-95:** %90.2 - IoU eşiği 0.5-0.95 aralığında ortalama doğruluk (genel başarı metriği)

---

### Sonuç

Model, **%90.2 mAP50-95** ve **%99.3 mAP50** ile son derece yüksek bir başarı oranına ulaşmıştır. Bu sonuçlar, modelin çatal ve kaşık nesnelerini yüksek doğrulukla tespit edebildiğini göstermektedir.